# Group XX - Evaluation Notebook
# Krones Bottle-Base Inspection — Inference + Runtime Measurement

**This is the ONE notebook the organizers run.** It runs top-to-bottom in a fresh Kaggle
environment with no manual steps. It:
1. loads our trained model + stacker + threshold from the **attached dataset** (`/kaggle/input/...`),
2. crops + preprocesses each test image exactly as in training,
3. runs the network (all folds) and the stacker to produce final predictions,
4. writes `submission.csv`,
5. **measures inference runtime** in a dedicated timing cell (the efficiency metric).

> **Setup before sharing:** attach the training artifacts as a Kaggle Dataset and set
> `MODEL_DIR` below to its path. The dataset must be shared (privately) with both organizers,
> or the model can't be loaded and the score is 0.


## 0 — Imports

In [ ]:
import os, sys, json, glob, time, statistics
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    import timm
except ImportError:
    os.system(f"{sys.executable} -m pip install -q timm"); import timm
try:
    import lightgbm as lgb
except ImportError:
    os.system(f"{sys.executable} -m pip install -q lightgbm"); import lightgbm as lgb

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 1 — Paths and loading the saved configuration

`MODEL_DIR` points at the attached dataset that holds our training artifacts. `best_threshold.json`
carries every setting (backbone, image size, threshold, mode, etc.) so this notebook stays in sync
with how the model was trained — we never hard-code those twice.

In [ ]:
# ── EDIT THIS to your attached dataset path ──
MODEL_DIR = Path("/kaggle/input/group-xx-krones-model")   # folder with model_fold*.pt, stacker.txt, best_threshold.json

# Competition data (organizers swap in the hidden test set here)
DATA_DIR = Path("/kaggle/input/competitions/1st-krones-vision-ai-challenge")
TEST_IMG_DIR = DATA_DIR/"test_images"
TEST_ANN     = DATA_DIR/"test_annotations_roi_only.json"
BOTTLETYPE   = DATA_DIR/"bottletypes.csv"

# Load the config saved by training
with open(MODEL_DIR/"best_threshold.json") as f:
    CFGJSON = json.load(f)
print("Loaded config:", json.dumps(CFGJSON, indent=2)[:600], "...")

BACKBONE  = CFGJSON["backbone"]; IMG_SIZE = CFGJSON["img_size"]; IN_CHANS = CFGJSON["in_chans"]
MEAN, STD = CFGJSON["mean"], CFGJSON["std"]; ROI_MARGIN = CFGJSON["roi_margin"]
N_AUX     = CFGJSON["n_aux"]; BOTTLE_TYPES = CFGJSON["types"]
USE_ADV   = CFGJSON["use_advanced_arch"]; USE_TTA = CFGJSON["use_tta"]
MODE      = CFGJSON["mode"]; THRESHOLD = CFGJSON["best_threshold"]
MODEL_FILES = CFGJSON["model_paths"]
print(f"\nbackbone={BACKBONE} img={IMG_SIZE} mode={MODE} thr={THRESHOLD} folds={len(MODEL_FILES)}")

## 2 — Model definition (must match training exactly)

We redefine the same `KronesNet` (and GeM/SE if the advanced arch was used) so the saved weights
load correctly. This is identical to the training notebook's model.

In [ ]:
class GeMPooling(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__(); self.p = nn.Parameter(torch.ones(1)*p); self.eps = eps
    def forward(self, x):
        p = self.p.clamp(1.0, 5.0)
        with torch.amp.autocast(device_type=x.device.type, enabled=False):
            xf = x.float().clamp(min=self.eps)
            out = F.avg_pool2d(xf.pow(p), kernel_size=(xf.shape[-2], xf.shape[-1])).pow(1.0/p)
        return out.view(out.shape[0], -1)

class SEGate(nn.Module):
    def __init__(self, feat, reduction=16):
        super().__init__(); mid = max(feat//reduction, 16)
        self.g = nn.Sequential(nn.Linear(feat, mid), nn.ReLU(inplace=True),
                               nn.Linear(mid, feat), nn.Sigmoid())
    def forward(self, x): return x * self.g(x)

class KronesNet(nn.Module):
    def __init__(self, pretrained=False):
        super().__init__()
        self.advanced = USE_ADV
        if self.advanced:
            self.encoder = timm.create_model(BACKBONE, pretrained=pretrained,
                                             in_chans=IN_CHANS, num_classes=0, global_pool="")
            feat = self.encoder.num_features
            self.gem = GeMPooling(); self.gate = SEGate(feat)
        else:
            self.encoder = timm.create_model(BACKBONE, pretrained=pretrained,
                                             in_chans=IN_CHANS, num_classes=0)
            feat = self.encoder.num_features
        self.head = nn.Linear(feat, 1); self.aux_head = nn.Linear(feat, N_AUX)
    def forward(self, x):
        f = self.gate(self.gem(self.encoder.forward_features(x))) if self.advanced else self.encoder(x)
        return self.head(f).squeeze(-1), self.aux_head(f)

## 3 — ROI crop + test dataset (identical preprocessing to training)

In [ ]:
def crop_roi(img, rx, ry, rw, rh, margin, out_size):
    h, w = img.shape[:2]
    cx, cy = rx + rw/2.0, ry + rh/2.0
    s = max(rw, rh) * (1.0 + 2.0*margin)
    x0, y0 = int(round(cx - s/2)), int(round(cy - s/2))
    x1, y1 = int(round(cx + s/2)), int(round(cy + s/2))
    px0, py0 = max(0,-x0), max(0,-y0); px1, py1 = max(0,x1-w), max(0,y1-h)
    crop = img[max(0,y0):min(h,y1), max(0,x0):min(w,x1)]
    if px0 or py0 or px1 or py1:
        crop = cv2.copyMakeBorder(crop, py0, py1, px0, px1, cv2.BORDER_REFLECT_101)
    interp = cv2.INTER_AREA if crop.shape[0] > out_size else cv2.INTER_LINEAR
    return cv2.resize(crop, (out_size, out_size), interpolation=interp)

# Test ROI boxes (the test set ships ROI-only annotations)
te_roi = {}
if TEST_ANN.exists():
    with open(TEST_ANN) as f: ta = json.load(f)
    te_id2file = {im["id"]: Path(im["file_name"]).name for im in ta["images"]}
    te_roi = {te_id2file[a["image_id"]]: a["bbox"] for a in ta["annotations"]}
# fallback if any image lacks an ROI
_r = np.array(list(te_roi.values()), np.float32) if te_roi else np.array([[160,147,987,722]],np.float32)
FALLBACK_ROI = tuple(np.median(_r, axis=0))

class TestDataset(Dataset):
    def __init__(self, ids, img_dir, roi_map):
        self.ids=list(ids); self.img_dir=Path(img_dir); self.roi=roi_map
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        f = self.ids[i]
        img = cv2.imread(str(self.img_dir/f), cv2.IMREAD_GRAYSCALE)
        rx,ry,rw,rh = self.roi.get(f, FALLBACK_ROI)
        img = crop_roi(img, rx,ry,rw,rh, ROI_MARGIN, IMG_SIZE)
        x = torch.from_numpy(np.ascontiguousarray(img)).float()
        x = x.div_(255.0).sub_(MEAN).div_(STD).unsqueeze(0)
        return x, f

## 4 — Build the test list and load all fold models + the stacker

In [ ]:
def seq_key(fn):
    tail = Path(fn).stem.split("_")[-1]
    return int(tail) if tail.isdigit() else fn
test_ids = sorted([Path(p).name for p in glob.glob(str(TEST_IMG_DIR/"*"))], key=seq_key)
print("Test images:", len(test_ids))

# Load fold models
models = []
for fn in MODEL_FILES:
    net = KronesNet(pretrained=False)
    net.load_state_dict(torch.load(MODEL_DIR/fn, map_location="cpu"))
    net.to(DEVICE).eval(); models.append(net)
print("Loaded", len(models), "fold models")

# Load the stacker + bottle-type map for the test set
stacker = lgb.Booster(model_file=str(MODEL_DIR/"stacker.txt"))
test_type_map = {}
if BOTTLETYPE.exists():
    bt = pd.read_csv(BOTTLETYPE)
    test_type_map = dict(zip(bt["image_id"], bt["bottle_type"]))

## 5 — Inference function

Runs every fold model on a batch, averages their main + aux probabilities (with optional TTA),
then the stacker combines `[logit(main), aux, bottle-type]` into the final probability. Returns
the per-image FAULTY probability under the chosen `MODE` (raw / stack / blend).

In [ ]:
def logit(p, eps=1e-6):
    p = np.clip(p, eps, 1-eps); return np.log(p/(1-p))

def _views(x):
    return [x, torch.flip(x,[3]), torch.flip(x,[2]), torch.flip(x,[2,3])] if USE_TTA else [x]

@torch.no_grad()
def run_inference(ids, batch_size=32):
    dl = DataLoader(TestDataset(ids, TEST_IMG_DIR, te_roi),
                    batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    main_all = np.zeros(len(ids)); aux_all = np.zeros((len(ids), N_AUX)); ptr = 0
    for x, _ in dl:
        x = x.to(DEVICE); bs = x.size(0)
        m_sum = np.zeros(bs); a_sum = np.zeros((bs, N_AUX))
        for net in models:                       # average across folds
            mv, av = [], []
            for v in _views(x):                  # average across TTA views
                with torch.amp.autocast(device_type=DEVICE.type, enabled=DEVICE.type=="cuda"):
                    ml, al = net(v)
                mv.append(torch.sigmoid(ml.float())); av.append(torch.sigmoid(al.float()))
            m_sum += torch.stack(mv).mean(0).cpu().numpy()
            a_sum += torch.stack(av).mean(0).cpu().numpy()
        main_all[ptr:ptr+bs] = m_sum/len(models)
        aux_all[ptr:ptr+bs]  = a_sum/len(models)
        ptr += bs

    # bottle-type one-hot for the stacker
    type_oh = np.zeros((len(ids), len(BOTTLE_TYPES)), dtype=np.float32)
    for i, f in enumerate(ids):
        t = test_type_map.get(f)
        if t in BOTTLE_TYPES: type_oh[i, BOTTLE_TYPES.index(t)] = 1.0

    X = np.column_stack([logit(main_all), aux_all, type_oh]).astype(np.float32)
    stacked = stacker.predict(X)
    if MODE == "raw":     return main_all
    if MODE == "stack":   return stacked
    return 0.5*main_all + 0.5*stacked            # blend

## 6 — Generate predictions and write submission.csv

In [ ]:
probs = run_inference(test_ids)
preds = (probs >= THRESHOLD).astype(int)
submission = pd.DataFrame({"image_id": test_ids, "target": preds})
submission.to_csv("submission.csv", index=False)
nf = int(submission["target"].sum())
print(f"submission.csv: {len(submission)} rows | FAULTY={nf} ({100*nf/len(submission):.1f}%)")
print(submission.head())

## 7 — ⏱️ Inference runtime measurement (the EFFICIENCY metric)

This is the timing cell the organizers use. It measures the **full inference pipeline** on the test
set: image load + ROI crop + preprocessing + model forward (all folds) + stacker. We warm up first
(so one-time CUDA/cuDNN setup isn't counted), then time the whole pass and report total + per-image.

In [ ]:
# Warm up (not timed) — triggers cuDNN autotuning + CUDA context so timing is steady
_ = run_inference(test_ids[:min(64, len(test_ids))])
if torch.cuda.is_available(): torch.cuda.synchronize()

# ── Timed full-test-set inference ──
t0 = time.perf_counter()
_probs = run_inference(test_ids)
if torch.cuda.is_available(): torch.cuda.synchronize()
total_s = time.perf_counter() - t0

n = len(test_ids)
per_img_ms = total_s/n*1000
print("="*60)
print(f"INFERENCE RUNTIME (full pipeline, {n} images)")
print("="*60)
print(f"Total time      : {total_s:.3f} s")
print(f"Per image       : {per_img_ms:.2f} ms")
print(f"Throughput      : {n/total_s:.1f} images/sec")
REQ = 70000/3600
print(f"Line-speed need : {REQ:.1f} img/s (70,000 bottles/hour)")
print(f"Headroom        : {(n/total_s)/REQ:.2f}x {'PASS' if n/total_s>=REQ else 'BELOW'}")

## 8 — Done

`submission.csv` is written and the runtime is printed above. This notebook ran end-to-end with no
manual steps, loading the model from the attached (shared) dataset — exactly as the rules require.